### 1 Leer .grip de 2020 retorcar los datos que correspondan y guardarlo en un .csv comprimido dentro de un gzip

## Codigo para la lectura/limpieza de los datos del Clima

In [2]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================
YEARS     = range(2020, 2025) 
# YEARS     = range(2025, 2026)
BASE_PATH = "."

# ============================================================
# ZONAS FOTOVOLTAICAS
# vars_instant: tcc (nubosidad), t2m (temperatura)
# vars_accum:   ssrd (radiación solar acumulada)
# ============================================================
ZONAS_SOLAR = {
    "fotovoltaica_MesetaNorte": {
        "lat": (40.5, 43.0), "lon": (-6.5, -2.5),
        "vars_instant": ["tcc", "t2m"],
        "vars_accum":   ["ssrd"],
    },
    "fotovoltaica_MesetaSur": {
        "lat": (38.2, 40.5), "lon": (-4.5, -1.0),
        "vars_instant": ["tcc", "t2m"],
        "vars_accum":   ["ssrd"],
    },
    "fotovoltaica_Oeste": {
        "lat": (38.0, 40.5), "lon": (-7.5, -4.5),
        "vars_instant": ["tcc", "t2m"],
        "vars_accum":   ["ssrd"],
    },
    "fotovoltaica_SurMurcia": {
        "lat": (36.0, 38.0), "lon": (-7.5, -0.6),
        "vars_instant": ["tcc", "t2m"],
        "vars_accum":   ["ssrd"],
    },
    "fotovoltaica_ValleEbro": {
        "lat": (40.5, 42.5), "lon": (-2.0, 0.5),
        "vars_instant": ["tcc", "t2m"],
        "vars_accum":   ["ssrd"],
    },
    "fotovoltaica_Levante": {
        "lat": (38.0, 40.5), "lon": (-1.0, 0.7),
        "vars_instant": ["tcc", "t2m"],
        "vars_accum":   ["ssrd"],
    },
}

# Pesos fotovoltaicos por año (% potencia instalada)
# Orden: 2020, 2021, 2022, 2023, 2024, 2025
# Para años fuera de rango se usa el peso de 2020
PESOS_SOLAR = {
    "fotovoltaica_MesetaNorte": {
        2020: 7.792207792,  2021: 7.264957265,  2022: 7.619047619,
        2023: 8.298319328,  2024: 9.214659686,  2025: 13.38912134,
    },
    "fotovoltaica_MesetaSur": {
        2020: 18.50649351,  2021: 21.15384615,  2022: 22.22222222,
        2023: 25.21008403,  2024: 23.66492147,  2025: 22.17573222,
    },
    "fotovoltaica_Oeste": {
        2020: 23.7012987,   2021: 26.6025641,   2022: 28.14814815,
        2023: 25.84033613,  2024: 25.13089005,  2025: 21.44351464,
    },
    "fotovoltaica_SurMurcia": {
        2020: 36.47186147,  2021: 31.1965812,   2022: 30.05291005,
        2023: 28.99159664,  2024: 30.68062827,  2025: 32.11297071,
    },
    "fotovoltaica_ValleEbro": {
        2020: 10.17316017,  2021: 11.0042735,   2022: 9.735449735,
        2023: 9.768907563,  2024: 9.633507853,  2025: 9.309623431,
    },
    "fotovoltaica_Levante": {
        2020: 3.354978355,  2021: 2.777777778,  2022: 2.222222222,
        2023: 1.890756303,  2024: 1.67539267,   2025: 1.569037657,
    },
}

# ============================================================
# ZONAS EÓLICAS
# vars_instant: u100, v100 (viento 100m), u10, v10 (viento 10m), sp (), t2m (temperatura)
# ============================================================
ZONAS_EOLICA = {
    "eolica_Noroeste": {
        "lat": (40.5, 44.5), "lon": (-9.5, -2.0),
        "vars_instant": ["u100", "v100", "u10", "v10", "sp", "t2m"],
        "vars_accum":   [],
    },
    "eolica_Nordeste": {
        "lat": (40.5, 44.5), "lon": (-2.0, 3.5),
        "vars_instant": ["u100", "v100", "u10", "v10", "sp", "t2m"],
        "vars_accum":   [],
    },
    "eolica_Sur": {
        "lat": (35.5, 38.0), "lon": (-7.5, -1.0),
        "vars_instant": ["u100", "v100", "u10", "v10", "sp", "t2m"],
        "vars_accum":   [],
    },
    "eolica_CentroInterior": {
            "lat": (38.0, 40.5), "lon": (-4.5, -1.0),
            "vars_instant": ["u100", "v100", "u10", "v10", "sp", "t2m"],
            "vars_accum":   [],
    },
    "eolica_LevanteMurcia": {
        "lat": (37.3, 40.5), "lon": (-1.0, 0.7),
        "vars_instant": ["u100", "v100", "u10", "v10", "sp", "t2m"],
        "vars_accum":   [],
    },
}

# Pesos eólicos por año (% potencia instalada)
PESOS_EOLICA = {
    "eolica_Noroeste": {
        2020: 41.73469388, 2021: 41.12014286, 2022: 40.07528618,
        2023: 37.929985,   2024: 39.97049789, 2025: 40.63321525,
    },
    "eolica_Nordeste": {
        2020: 25.40816327, 2021: 25.9098141,  2022: 26.3596858,
        2023: 27.93543133, 2024: 27.68239123, 2025: 27.85749225,
    },
    "eolica_Sur": {
        2020: 12.85714286, 2021: 12.51886559, 2022: 12.23083073,
        2023: 12.38214491, 2024: 11.66145625, 2025: 11.45479893,
    },
    "eolica_CentroInterior": {
            2020: 14.3877551, 2021: 15.07472232, 2022: 16.2191845,
            2023: 16.6164767, 2024: 15.87837162, 2025: 15.39712935,
    },
    "eolica_LevanteMurcia": {
        2020: 5.612244898, 2021: 5.376455135, 2022: 5.115012782,
        2023: 5.13596205,  2024: 4.807283015, 2025: 4.657364217,
    },
}

# ============================================================
# ZONAS HIDRÁULICAS
# vars_accum: tp (precipitación → llenado embalses)
# Pesos fijos — potencia instalada no cambió 2020-2025
# ============================================================
ZONAS_HIDRO = {
    "hidraulica_Noroeste":    {"lat": (41.0, 44.0), "lon": (-9.5, -6.5),
                               "vars_instant": [], "vars_accum": ["tp"]},
    "hidraulica_Cantabrico":  {"lat": (42.8, 43.8), "lon": (-6.5, -3.5),
                               "vars_instant": [], "vars_accum": ["tp"]},
    "hidraulica_DueroMeseta": {"lat": (40.5, 43.0), "lon": (-6.5, -2.5),
                               "vars_instant": [], "vars_accum": ["tp"]},
    "hidraulica_TajoCentral": {"lat": (39.0, 41.2), "lon": (-7.5, -1.5),
                               "vars_instant": [], "vars_accum": ["tp"]},
    "hidraulica_JucarLevante":{"lat": (38.5, 40.5), "lon": (-2.5, 0.5),
                               "vars_instant": [], "vars_accum": ["tp"]},
    "hidraulica_SurBeticas":  {"lat": (36.0, 38.5), "lon": (-6.5, -2.0),
                               "vars_instant": [], "vars_accum": ["tp"]},
    "hidraulica_EbroPirineos":{"lat": (41.0, 43.5), "lon": (-1.0, 3.5),
                               "vars_instant": [], "vars_accum": ["tp"]},
}

PESOS_HIDRO = {
    "hidraulica_Noroeste":    22.19626168,
    "hidraulica_Cantabrico":   5.383990248,
    "hidraulica_DueroMeseta": 26.15806583,
    "hidraulica_TajoCentral": 14.19138562,
    "hidraulica_JucarLevante": 7.679804957,
    "hidraulica_SurBeticas":   3.677366924,
    "hidraulica_EbroPirineos": 20.71312475,
}

# ============================================================
# [NUEVO] PUNTOS FIJOS DE DEMANDA — 46 CAPITALES DE PROVINCIA
# Coordenadas GPS exactas → píxel ERA5 más cercano (method='nearest')
# Sin solapamientos, sin multicolinealidad
# ============================================================
PUNTOS_DEMANDA = {
    # Nodo Centro & Castilla-La Mancha
    "madrid":        {"lat": 40.4167, "lon": -3.7037},
    "toledo":        {"lat": 39.8628, "lon": -4.0273},
    "ciudad_real":   {"lat": 38.9848, "lon": -3.9273},
    "albacete":      {"lat": 38.9943, "lon": -1.8585},
    "guadalajara":   {"lat": 40.6327, "lon": -3.1643},
    "cuenca":        {"lat": 40.0704, "lon": -2.1374},
    # Cataluña
    "barcelona":     {"lat": 41.3851, "lon":  2.1734},
    "tarragona":     {"lat": 41.1189, "lon":  1.2445},
    "girona":        {"lat": 41.9794, "lon":  2.8214},
    "lleida":        {"lat": 41.6176, "lon":  0.6200},
    # Andalucía
    "sevilla":       {"lat": 37.3891, "lon": -5.9845},
    "malaga":        {"lat": 36.7213, "lon": -4.4214},
    "cadiz":         {"lat": 36.5271, "lon": -6.2886},
    "granada":       {"lat": 37.1773, "lon": -3.5986},
    "cordoba":       {"lat": 37.8882, "lon": -4.7794},
    "almeria":       {"lat": 36.8340, "lon": -2.4637},
    "jaen":          {"lat": 37.7796, "lon": -3.7849},
    "huelva":        {"lat": 37.2614, "lon": -6.9447},
    # Levante & Murcia
    "valencia":      {"lat": 39.4699, "lon": -0.3763},
    "alicante":      {"lat": 38.3452, "lon": -0.4810},
    "castellon":     {"lat": 39.9864, "lon": -0.0513},
    "murcia":        {"lat": 37.9922, "lon": -1.1307},
    # Galicia
    "coruna":        {"lat": 43.3623, "lon": -8.4115},
    "pontevedra":    {"lat": 42.4310, "lon": -8.6444},
    "lugo":          {"lat": 43.0097, "lon": -7.5568},
    "ourense":       {"lat": 42.3358, "lon": -7.8639},
    # Castilla y León
    "valladolid":    {"lat": 41.6523, "lon": -4.7245},
    "leon":          {"lat": 42.5987, "lon": -5.5671},
    "burgos":        {"lat": 42.3440, "lon": -3.6969},
    "salamanca":     {"lat": 40.9688, "lon": -5.6639},
    "zamora":        {"lat": 41.5063, "lon": -5.7446},
    "palencia":      {"lat": 42.0096, "lon": -4.5284},
    "segovia":       {"lat": 40.9429, "lon": -4.1088},
    "avila":         {"lat": 40.6567, "lon": -4.6811},
    "soria":         {"lat": 41.7660, "lon": -2.4683},
    # Norte & Cornisa Cantábrica
    "bilbao":        {"lat": 43.2630, "lon": -2.9350},
    "san_sebastian": {"lat": 43.3183, "lon": -1.9812},
    "vitoria":       {"lat": 42.8467, "lon": -2.6717},
    "oviedo":        {"lat": 43.3614, "lon": -5.8502},
    "santander":     {"lat": 43.4623, "lon": -3.8058},
    # Ebro, Extremadura y Resto
    "zaragoza":      {"lat": 41.6488, "lon": -0.8891},
    "huesca":        {"lat": 42.1362, "lon": -0.4084},
    "teruel":        {"lat": 40.3457, "lon": -1.1065},
    "badajoz":       {"lat": 38.8794, "lon": -6.9706},
    "caceres":       {"lat": 39.4753, "lon": -6.3722},
    "pamplona":      {"lat": 42.8125, "lon": -1.6458},
    "logrono":       {"lat": 42.4627, "lon": -2.4450},
}

# ============================================================
# [NUEVO] PESOS POBLACIONALES DINÁMICOS POR AÑO (% padrón INE)
# Cambian año a año → refleja crecimiento/despoblación real
# Suma = 100.0 por año en la Península Ibérica
# Para años fuera del rango 2020-2025 se usa el peso de 2020
# ============================================================
PESOS_POBLACION = {
   
    "madrid":       {2020: 15.3341, 2021: 15.3664, 2022: 15.4005, 2023: 15.4408, 2024: 15.4822, 2025: 15.4356},
    "barcelona":    {2020: 13.1924, 2021: 13.1464, 2022: 13.1070, 2023: 13.0631, 2024: 13.0729, 2025: 13.0446},
    "valencia":     {2020:  5.9097, 2021:  5.9202, 2022:  5.9320, 2023:  5.9438, 2024:  5.9496, 2025:  5.9766},
    "sevilla":      {2020:  4.4534, 2021:  4.4545, 2022:  4.4520, 2023:  4.4381, 2024:  4.4259, 2025:  4.4039},
    "alicante":     {2020:  4.2647, 2021:  4.2819, 2022:  4.3065, 2023:  4.3376, 2024:  4.3688, 2025:  4.3885},
    "malaga":       {2020:  3.8114, 2021:  3.8374, 2022:  3.8686, 2023:  3.9071, 2024:  3.9348, 2025:  3.9413},
    "murcia":       {2020:  3.4446, 2021:  3.4448, 2022:  3.4485, 2023:  3.4614, 2024:  3.4759, 2025:  3.4932},
    "cadiz":        {2020:  2.8399, 2021:  2.8420, 2022:  2.8386, 2023:  2.8329, 2024:  2.8190, 2025:  2.8138},
    "bilbao":       {2020:  2.6361, 2021:  2.6293, 2022:  2.6233, 2023:  2.6130, 2024:  2.6037, 2025:  2.5967},
    "coruna":       {2020:  2.5661, 2021:  2.5627, 2022:  2.5557, 2023:  2.5457, 2024:  2.5340, 2025:  2.5289},
    "oviedo":       {2020:  2.3139, 2021:  2.3031, 2022:  2.2935, 2023:  2.2794, 2024:  2.2656, 2025:  2.2623},
    "zaragoza":     {2020:  2.2129, 2021:  2.2134, 2022:  2.2106, 2023:  2.2027, 2024:  2.1932, 2025:  2.1766},
    "pontevedra":   {2020:  2.1523, 2021:  2.1505, 2022:  2.1460, 2023:  2.1380, 2024:  2.1298, 2025:  2.1247},
    "granada":      {2020:  2.1058, 2021:  2.1092, 2022:  2.1064, 2023:  2.1020, 2024:  2.0945, 2025:  2.0977},
    "tarragona":    {2020:  1.8525, 2021:  1.8628, 2022:  1.8721, 2023:  1.8871, 2024:  1.9023, 2025:  1.9094},
    "girona":       {2020:  1.7582, 2021:  1.7681, 2022:  1.7774, 2023:  1.7962, 2024:  1.8098, 2025:  1.8196},
    "cordoba":      {2020:  1.7815, 2021:  1.7752, 2022:  1.7662, 2023:  1.7548, 2024:  1.7423, 2025:  1.7380},
    "almeria":      {2020:  1.6373, 2021:  1.6510, 2022:  1.6672, 2023:  1.6856, 2024:  1.6951, 2025:  1.6951},
    "toledo":       {2020:  1.5798, 2021:  1.5963, 2022:  1.6179, 2023:  1.6350, 2024:  1.6505, 2025:  1.6541},
    "san_sebastian":{2020:  1.6573, 2021:  1.6541, 2022:  1.6500, 2023:  1.6439, 2024:  1.6376, 2025:  1.6336},
    "pamplona":     {2020:  1.4957, 2021:  1.5005, 2022:  1.5056, 2023:  1.5110, 2024:  1.5129, 2025:  1.5124},
    "badajoz":      {2020:  1.5363, 2021:  1.5309, 2022:  1.5240, 2023:  1.5140, 2024:  1.5036, 2025:  1.4986},
    "jaen":         {2020:  1.4354, 2021:  1.4259, 2022:  1.4151, 2023:  1.4029, 2024:  1.3941, 2025:  1.3965},
    "castellon":    {2020:  1.3252, 2021:  1.3290, 2022:  1.3342, 2023:  1.3416, 2024:  1.3496, 2025:  1.3598},
    "santander":    {2020:  1.3283, 2021:  1.3267, 2022:  1.3231, 2023:  1.3205, 2024:  1.3207, 2025:  1.3239},
    "huelva":       {2020:  1.1970, 2021:  1.2007, 2022:  1.1989, 2023:  1.1984, 2024:  1.1987, 2025:  1.1957},
    "valladolid":   {2020:  1.1879, 2021:  1.1824, 2022:  1.1798, 2023:  1.1752, 2024:  1.1687, 2025:  1.1730},
    "ciudad_real":  {2020:  1.1254, 2021:  1.1190, 2022:  1.1131, 2023:  1.1057, 2024:  1.0981, 2025:  1.1053},
    "leon":         {2020:  1.0446, 2021:  1.0362, 2022:  1.0283, 2023:  1.0199, 2024:  1.0087, 2025:  1.0093},
    "lleida":       {2020:  1.0009, 2021:  1.0029, 2022:  1.0061, 2023:  1.0096, 2024:  1.0087, 2025:  1.0066},
    "albacete":     {2020:  0.8898, 2021:  0.8878, 2022:  0.8839, 2023:  0.8797, 2024:  0.8770, 2025:  0.8759},
    "caceres":      {2020:  0.8890, 2021:  0.8849, 2022:  0.8796, 2023:  0.8758, 2024:  0.8719, 2025:  0.8726},
    "burgos":       {2020:  0.8124, 2021:  0.8088, 2022:  0.8062, 2023:  0.8061, 2024:  0.8042, 2025:  0.8041},
    "vitoria":      {2020:  0.7586, 2021:  0.7576, 2022:  0.7568, 2023:  0.7548, 2024:  0.7539, 2025:  0.7525},
    "salamanca":    {2020:  0.7485, 2021:  0.7424, 2022:  0.7365, 2023:  0.7326, 2024:  0.7280, 2025:  0.7326},
    "lugo":         {2020:  0.7474, 2021:  0.7424, 2022:  0.7373, 2023:  0.7315, 2024:  0.7259, 2025:  0.7290},
    "logrono":      {2020:  0.7316, 2021:  0.7300, 2022:  0.7284, 2023:  0.7271, 2024:  0.7249, 2025:  0.7255},
    "ourense":      {2020:  0.6980, 2021:  0.6919, 2022:  0.6861, 2023:  0.6811, 2024:  0.6754, 2025:  0.6845},
    "guadalajara":  {2020:  0.5970, 2021:  0.6071, 2022:  0.6183, 2023:  0.6247, 2024:  0.6253, 2025:  0.6130},
    "huesca":       {2020:  0.5144, 2021:  0.5133, 2022:  0.5123, 2023:  0.5114, 2024:  0.5097, 2025:  0.5105},
    "cuenca":       {2020:  0.4529, 2021:  0.4477, 2022:  0.4438, 2023:  0.4411, 2024:  0.4381, 2025:  0.4465},
    "zamora":       {2020:  0.3883, 2021:  0.3831, 2022:  0.3750, 2023:  0.3685, 2024:  0.3635, 2025:  0.3756},
    "avila":        {2020:  0.3647, 2021:  0.3618, 2022:  0.3576, 2023:  0.3542, 2024:  0.3524, 2025:  0.3558},
    "palencia":     {2020:  0.3629, 2021:  0.3579, 2022:  0.3536, 2023:  0.3514, 2024:  0.3487, 2025:  0.3539},
    "segovia":      {2020:  0.3529, 2021:  0.3518, 2022:  0.3509, 2023:  0.3499, 2024:  0.3490, 2025:  0.3495},
    "teruel":       {2020:  0.3075, 2021:  0.3053, 2022:  0.3034, 2023:  0.3019, 2024:  0.3004, 2025:  0.3018},
    "soria":        {2020:  0.2035, 2021:  0.2027, 2022:  0.2021, 2023:  0.2012, 2024:  0.2004, 2025:  0.2014},
}

# Umbrales térmicos para el mercado ibérico
T_REF_FRIO  = 19.0  # °C — por debajo activa calefacción 
T_REF_CALOR = 27.0  # °C — por encima activa refrigeración

# Constante del gas para aire seco (J/kg/K)
# Usada en: ρ = sp / (R_AIRE · T)
R_AIRE = 287.05

# ============================================================
# FUNCIONES AUXILIARES — GENERALES
# ============================================================

def limpiar_idx(carpeta):
    """Borra índices .idx corruptos generados por cfgrib."""
    for f in glob.glob(os.path.join(carpeta, '*.idx')):
        os.remove(f)


def recortar_zona(ds, lat_min, lat_max, lon_min, lon_max):
    """
    Recorta dataset a una bbox.
    ERA5 va de N→S → slice invertido en latitud.
    Devuelve None si la zona queda vacía.
    """
    ds_zona = ds.sel(
        latitude=slice(lat_max, lat_min),
        longitude=slice(lon_min, lon_max)
    )
    if ds_zona.dims.get('latitude', 0) == 0 or \
       ds_zona.dims.get('longitude', 0) == 0:
        return None
    return ds_zona


def normalizar_tiempo(df):
    """Asegura que la columna temporal se llama 'time'."""
    if 'time' not in df.columns and 'valid_time' in df.columns:
        df = df.rename(columns={'valid_time': 'time'})
    return df


def get_peso_energia(zona, year, pesos_dict):
    """
    Obtiene el peso energético (solar/eólico) de una zona para un año.
    Para años fuera del rango usa el peso de 2020.
    Convierte % → proporción dividiendo entre 100.
    """
    pesos_zona = pesos_dict.get(zona, {})
    if isinstance(pesos_zona, dict):
        anio = year if year in pesos_zona else min(pesos_zona.keys(),
                                                    key=lambda a: abs(a - year))
        return pesos_zona[anio] / 100.0
    return pesos_zona / 100.0


def get_peso_poblacion(ciudad, year):
    """
    Obtiene el peso poblacional de una ciudad para un año.
    Para años fuera del rango 2020-2025 usa el año más cercano.
    Devuelve proporción (no porcentaje).
    """
    pesos_ciudad = PESOS_POBLACION.get(ciudad, {})
    if year in pesos_ciudad:
        return pesos_ciudad[year] / 100.0
    if not pesos_ciudad:
        return 0.0
    anio_cercano = min(pesos_ciudad.keys(), key=lambda a: abs(a - year))
    return pesos_ciudad[anio_cercano] / 100.0


# ============================================================
# FUNCIONES DE CARGA DE GRIB
# ============================================================

def cargar_instant(path):
    """
    Carga variables instantáneas del GRIB.
    Usa valid_time como índice → 24 timestamps/día correctos.
    Variables: t2m, tcc, u10, v10, u100, v100, sp
    """
    try:
        ds = xr.open_dataset(
            path,
            engine='cfgrib',
            backend_kwargs={'filter_by_keys': {'stepType': 'instant'}}
        )
        if 'valid_time' in ds.coords:
            ds = ds.assign_coords(time=ds.valid_time)
        coords_extra = [c for c in ['step', 'valid_time', 'number', 'surface']
                        if c in ds.coords]
        ds = ds.drop_vars(coords_extra)
        return ds
    except Exception as e:
        print(f"  ⚠️ Error cargando instant: {e}")
        return None


def cargar_accum(path, year):
    """
    Carga variables acumuladas del GRIB (ssrd, tp).
    ERA5 las guarda con estructura 2D (time × step).
    Stack aplana (time, step) → valid_time 1D horario.
    Filtra solo el año solicitado.
    """
    try:
        ds = xr.open_dataset(
            path,
            engine='cfgrib',
            backend_kwargs={'filter_by_keys': {'stepType': 'accum'}}
        )
        ds_stack = ds.stack(valid_time_1d=('time', 'step'))
        ds_stack = ds_stack.assign_coords(
            valid_time_1d=ds_stack.valid_time.values
        )
        ds_out = ds_stack.rename({'valid_time_1d': 'time'})
        coords_extra = [c for c in ['step', 'valid_time', 'number', 'surface']
                        if c in ds_out.coords]
        ds_out = ds_out.drop_vars(coords_extra)
        ds_out = ds_out.sortby('time')
        ds_out = ds_out.sel(time=ds_out.time.dt.year == year)
        print(f"     Timestamps accum año {year}: {ds_out.dims['time']}")
        return ds_out
    except Exception as e:
        print(f"  ⚠️ Error cargando accum: {e}")
        return None


# ============================================================
# FUNCIONES DE PROCESADO — ENERGÍA SOLAR Y EÓLICA
# ============================================================

def suma_ponderada(ds, zonas_dict, pesos_dict, year, vars_lista, prefijo_col):
    """
    Calcula la media ponderada espacial entre zonas.
    Para cada zona: extrae media espacial × peso del año.
    Suma todas las zonas → un único valor por hora.

    Parámetros:
      ds          : dataset xarray
      zonas_dict  : diccionario de zonas geográficas
      pesos_dict  : {zona: {año: peso}} o {zona: peso}
      year        : año para seleccionar el peso correcto
      vars_lista  : variables a extraer (ej: ["ssrd"])
      prefijo_col : sufijo de las columnas de salida (ej: "solar")

    Retorna DataFrame con ['time'] + ['{var}_{prefijo}' × vars]
    """
    acumulado  = None
    suma_pesos = 0.0

    for zona_nombre, zona_cfg in zonas_dict.items():
        peso = get_peso_energia(zona_nombre, year, pesos_dict)
        if peso == 0:
            continue

        lat_min, lat_max = zona_cfg["lat"]
        lon_min, lon_max = zona_cfg["lon"]
        ds_zona = recortar_zona(ds, lat_min, lat_max, lon_min, lon_max)
        if ds_zona is None:
            continue

        vars_ok = [v for v in vars_lista if v in ds_zona.data_vars]
        if not vars_ok:
            continue

        media_zona = ds_zona[vars_ok].mean(dim=['latitude', 'longitude'])

        if acumulado is None:
            acumulado = media_zona * peso
        else:
            acumulado = acumulado + media_zona * peso

        suma_pesos += peso

    if acumulado is None:
        return None

    # Normalizar por suma de pesos → media ponderada real
    acumulado = acumulado / suma_pesos

    df = acumulado.to_dataframe().reset_index()
    df = normalizar_tiempo(df)

    rename = {v: f"{v}_{prefijo_col}" for v in vars_lista if v in df.columns}
    df = df.rename(columns=rename)

    cols = ['time'] + [f"{v}_{prefijo_col}" for v in vars_lista
                       if f"{v}_{prefijo_col}" in df.columns]
    return df[cols].dropna()


# ============================================================
# [NUEVO] FUNCIONES DE PROCESADO — DEMANDA (HDD/CDD)
# Reemplaza la anterior extraer_demanda_ponderada()
# ============================================================

def extraer_temp_puntos_fijos(ds_instant):
    """
    [OPTIMIZADO] Extrae temperatura de las 46 capitales en una
    sola operación vectorizada usando xarray.sel() con listas
    de coordenadas. Evita los 46 merges secuenciales.
    """
    if 't2m' not in ds_instant.data_vars:
        return None

    ciudades  = list(PUNTOS_DEMANDA.keys())
    lats      = [PUNTOS_DEMANDA[c]["lat"] for c in ciudades]
    lons      = [PUNTOS_DEMANDA[c]["lon"] for c in ciudades]

    try:
        # Seleccionar todas las ciudades a la vez → una sola operación
        # Resultado shape: (time, ciudad) en vez de 46 × (time,)
        puntos = ds_instant['t2m'].sel(
            latitude=xr.DataArray(lats, dims='ciudad'),
            longitude=xr.DataArray(lons, dims='ciudad'),
            method='nearest'
        )

        # Convertir a DataFrame en una sola operación
        df = puntos.to_dataframe(name='t2m').reset_index()

        # Normalizar columna temporal
        if 'time' not in df.columns and 'valid_time' in df.columns:
            df = df.rename(columns={'valid_time': 'time'})

        # Kelvin → Celsius
        df['t2m'] = df['t2m'] - 273.15

        # Añadir nombres de ciudad usando el índice de la dimensión ciudad
        # ciudad_idx va de 0 a 45 → mapeamos a nombre
        if 'ciudad' in df.columns:
            idx_to_ciudad = {i: c for i, c in enumerate(ciudades)}
            df['ciudad_nombre'] = df['ciudad'].map(idx_to_ciudad)
        else:
            # Si xarray ya resolvió las coordenadas, reconstruir desde lat/lon
            lat_to_ciudad = {PUNTOS_DEMANDA[c]["lat"]: c for c in ciudades}
            df['ciudad_nombre'] = df['latitude'].map(lat_to_ciudad)

        # Pivotar: (time, ciudad) → una columna por ciudad
        df_pivot = df.pivot_table(
            index='time',
            columns='ciudad_nombre',
            values='t2m',
            aggfunc='first'
        ).reset_index()

        # Renombrar columnas: ciudad → t2m_ciudad
        rename = {c: f't2m_{c}' for c in ciudades if c in df_pivot.columns}
        df_pivot = df_pivot.rename(columns=rename)
        df_pivot.columns.name = None

        print(f"     ✅ {len(ciudades)} capitales extraídas — {df_pivot.shape}")
        return df_pivot.dropna()

    except Exception as e:
        print(f"  ⚠️ Error en extracción vectorizada: {e}")
        print(f"     Intentando fallback secuencial (más lento)...")

        # Fallback: si falla el método vectorizado, hacerlo
        # secuencialmente pero con concat en vez de merge
        registros = {}
        time_index = None

        for ciudad, coords in PUNTOS_DEMANDA.items():
            try:
                punto = ds_instant['t2m'].sel(
                    latitude=coords["lat"],
                    longitude=coords["lon"],
                    method='nearest'
                )
                vals = punto.values - 273.15  # Kelvin → Celsius
                registros[f't2m_{ciudad}'] = vals
                if time_index is None:
                    time_vals = punto.coords.get(
                        'valid_time',
                        punto.coords.get('time', None)
                    )
                    if time_vals is not None:
                        time_index = time_vals.values

            except Exception as e2:
                print(f"     ⚠️ {ciudad}: {e2}")

        if not registros or time_index is None:
            return None

        # Construir DataFrame de golpe → sin merges
        df_out = pd.DataFrame(registros)
        df_out.insert(0, 'time', time_index)
        print(f"     ✅ Fallback OK — {df_out.shape}")
        return df_out.dropna()


def calcular_hdd_cdd(df_temps, year):
    """
    [NUEVO] Calcula HDD y CDD horarios y los fusiona en
    tres columnas ponderadas usando los pesos del INE.

    Las 46 temperaturas individuales se colapsan en:
      - hdd_peninsular_ponderado : estrés por frío nacional
      - cdd_peninsular_ponderado : estrés por calor nacional
      - estres_termico_total     : suma de ambos (U-shape resuelta)

    HDD horario = max(0, 18 - T) / 24
    CDD horario = max(0, T - 22) / 24
    División /24: normaliza hora → escala diaria.

    Los pesos varían por año (padrón INE):
      - Madrid/Málaga/Alicante: crecen → más peso en el pool
      - Zamora/Soria/Palencia:  decrecen → menos peso en el pool

    Parámetros:
        df_temps : DataFrame con 't2m_{ciudad}' por cada capital
        year     : año para seleccionar los pesos INE correctos

    Retorna DataFrame con SOLO las 3 columnas agregadas
    (las temperaturas individuales se descartan → menos columnas)
    """
    df = df_temps.copy()

    hdd_pond = pd.Series(0.0, index=df.index)
    cdd_pond = pd.Series(0.0, index=df.index)
    suma_pesos = 0.0

    for ciudad in PUNTOS_DEMANDA.keys():
        col = f't2m_{ciudad}'
        if col not in df.columns:
            continue

        peso = get_peso_poblacion(ciudad, year)

        # HDD horario — estrés por frío
        hdd_ciudad = np.maximum(0, T_REF_FRIO - df[col]) / 24.0
        # CDD horario — estrés por calor
        cdd_ciudad = np.maximum(0, df[col] - T_REF_CALOR) / 24.0

        hdd_pond += hdd_ciudad * peso
        cdd_pond += cdd_ciudad * peso
        suma_pesos += peso

    # Verificar integridad de los pesos del INE
    print(f"     Suma pesos poblacionales {year}: {suma_pesos:.4f} "
          f"({'✅' if abs(suma_pesos - 1.0) < 0.01 else '⚠️ revisar pesos'})")

    # Construir DataFrame de salida SOLO con las 3 columnas agregadas
    # Las temperaturas individuales no se guardan → dataset compacto
    df_out = df[['time']].copy()
    df_out['hdd_peninsular_ponderado'] = hdd_pond.values
    df_out['cdd_peninsular_ponderado'] = cdd_pond.values
    df_out['estres_termico_total']     = (hdd_pond + cdd_pond).values

    return df_out.dropna()


# ============================================================
# PROCESAR AÑO COMPLETO
# ============================================================

def procesar_año(year):
    carpeta = os.path.join(BASE_PATH, str(year))
    print(f"\n{'='*60}")
    print(f"🔬 Procesando año {year}")
    print(f"{'='*60}")

    limpiar_idx(carpeta)

    path = os.path.join(carpeta, f"data{year}DatosClima.grib")
    if not os.path.exists(path):
        print(f"  ❌ No encontrado: {path}")
        return None

    dfs = []

    # ----------------------------------------------------------
    # INSTANT — t2m, tcc, u10, v10, u100, v100
    # ----------------------------------------------------------
    print("  📂 Cargando instant...")
    ds_instant = cargar_instant(path)

    if ds_instant is not None:
        print(f"     Variables: {list(ds_instant.data_vars)}")
        print(f"     Timestamps: {ds_instant.dims.get('time', '?')}")

        # ── SOLAR instant: tcc y t2m ponderados por potencia ──
        for var in ["tcc", "t2m"]:
            if var not in ds_instant.data_vars:
                continue
            df_v = suma_ponderada(
                ds_instant, ZONAS_SOLAR, PESOS_SOLAR,
                year, [var], "solar"
            )
            if df_v is not None:
                print(f"     ✅ solar {var} ponderado — {df_v.shape}")
                dfs.append(df_v)

        # ── EÓLICA instant: u100, v100, u10, v10, sp, t2m ponderados ──
        for var in ["u100", "v100", "u10", "v10", "sp", "t2m"]:
            if var not in ds_instant.data_vars:
                continue
            df_v = suma_ponderada(
                ds_instant, ZONAS_EOLICA, PESOS_EOLICA,
                year, [var], "eolica"
            )
            if df_v is not None:
                print(f"     ✅ eólica {var} ponderado — {df_v.shape}")
                dfs.append(df_v)

        # ── [NUEVO] DEMANDA: 46 capitales → HDD/CDD ponderados ──
        # Reemplaza la anterior extraer_demanda_ponderada()
        # Las 46 temperaturas se colapsan en 3 columnas finales
        df_temps = extraer_temp_puntos_fijos(ds_instant)
        if df_temps is not None:
            df_hdd_cdd = calcular_hdd_cdd(df_temps, year)
            if df_hdd_cdd is not None:
                print(f"     ✅ HDD/CDD peninsular ({year}) — {df_hdd_cdd.shape}")
                dfs.append(df_hdd_cdd)

        del ds_instant

    # ----------------------------------------------------------
    # ACCUM — ssrd (solar), tp (hidráulica)
    # ----------------------------------------------------------
    print("  📂 Cargando accum...")
    ds_accum = cargar_accum(path, year)

    if ds_accum is not None:
        print(f"     Variables: {list(ds_accum.data_vars)}")

        # ── SOLAR accum: ssrd ponderado por potencia ──
        if 'ssrd' in ds_accum.data_vars:
            df_ssrd = suma_ponderada(
                ds_accum, ZONAS_SOLAR, PESOS_SOLAR,
                year, ["ssrd"], "solar"
            )
            if df_ssrd is not None:
                print(f"     ✅ solar ssrd ponderado — {df_ssrd.shape}")
                dfs.append(df_ssrd)

        # ── HIDRÁULICA accum: tp ponderado por potencia ──
        if 'tp' in ds_accum.data_vars:
            df_tp = suma_ponderada(
                ds_accum, ZONAS_HIDRO, PESOS_HIDRO,
                year, ["tp"], "hidro"
            )
            if df_tp is not None:
                print(f"     ✅ hidráulica tp ponderado — {df_tp.shape}")
                dfs.append(df_tp)

        del ds_accum

    if not dfs:
        print(f"  ❌ Sin datos para {year}")
        return None

    # ----------------------------------------------------------
    # MERGE PROGRESIVO POR TIME
    # ----------------------------------------------------------
    df = dfs[0]
    for df_z in dfs[1:]:
        df = pd.merge(df, df_z, on='time', how='inner')
    df = df.dropna()

    print(f"\n  📊 Tras merge — shape: {df.shape}")
    print(f"  🕐 Timestamps únicos: {df['time'].nunique()}")
    print(f"  📅 Rango: {df['time'].min()} → {df['time'].max()}")

    # ----------------------------------------------------------
    # FEATURE ENGINEERING
    # ----------------------------------------------------------

    # Velocidad viento 100m (aerogeneradores)
    if 'u100_eolica' in df.columns and 'v100_eolica' in df.columns:
        df['wind_speed_100m_ponderado'] = np.sqrt(
            df['u100_eolica']**2 + df['v100_eolica']**2
        )

    # Velocidad viento 10m
    if 'u10_eolica' in df.columns and 'v10_eolica' in df.columns:
        df['wind_speed_10m_ponderado'] = np.sqrt(
            df['u10_eolica']**2 + df['v10_eolica']**2
        )

    # Wind shear — gradiente vertical (capeado a 5)
    if 'wind_speed_10m_ponderado' in df.columns and \
       'wind_speed_100m_ponderado' in df.columns:
        df['wind_shear_ponderado'] = (
            df['wind_speed_100m_ponderado'] /
            df['wind_speed_10m_ponderado'].replace(0, np.nan)
        ).clip(upper=5)

    # [NUEVO] Densidad del aire — ecuación de gas ideal
    # ρ = sp / (R_aire · T)
    # R_aire = 287.05 J/(kg·K) constante del gas para aire seco
    # t2m viene en Kelvin desde ERA5 — NO convertir a Celsius aquí

    if 'sp_eolica' in df.columns and 't2m_eolica' in df.columns:
        df['densidad_aire_eolica'] = (
            df['sp_eolica'] / (R_AIRE * df['t2m_eolica'])
        )
        print(f"  ✅ densidad_aire_eolica calculada")
        print(f"     Rango: [{df['densidad_aire_eolica'].min():.3f}, "
            f"{df['densidad_aire_eolica'].max():.3f}] kg/m³")

    # [NUEVO] Potencia eólica específica corregida por densidad
    # P_norm = ½ · ρ · v³
    # No incluimos A ni Cp porque son constantes del parque,
    # no del clima. El modelo aprende la relación directamente.
    # Unidades: W/m² (potencia por unidad de área de rotor)
    if 'densidad_aire_eolica' in df.columns and \
    'wind_speed_100m_ponderado' in df.columns:
        df['wind_power_density_ponderado'] = (
            0.5
            * df['densidad_aire_eolica']
            * df['wind_speed_100m_ponderado']**3
        )
        print(f"  ✅ wind_power_density_ponderado calculado")

    # Eliminar componentes U/V y variables intermedias
    # sp y t2m eólicos ya no se necesitan tras calcular densidad
    cols_eliminar = [c for c in df.columns
                    if c.startswith(('u100_', 'v100_', 'u10_', 'v10_',
                                    'sp_eolica', 't2m_eolica'))]
    df = df.drop(columns=cols_eliminar)
    

    # Temperatura solar: Kelvin → Celsius
    if 't2m_solar' in df.columns:
        df['t2m_celsius_solar_ponderado'] = df['t2m_solar'] - 273.15
        df = df.drop(columns=['t2m_solar'])

    # Renombrar columnas a nombres finales limpios
    rename_final = {
        'ssrd_solar': 'ssrd_solar_ponderado',
        'tcc_solar':  'tcc_solar_ponderado',
        'tp_hidro':   'tp_hidro_ponderado',
    }
    df = df.rename(columns={k: v for k, v in rename_final.items()
                             if k in df.columns})

    print(f"  ✅ Shape final: {df.shape}")
    print(f"  📋 Columnas: {df.columns.tolist()}")

    # ----------------------------------------------------------
    # GUARDAR CSV COMPRIMIDO
    # ----------------------------------------------------------
    output_path = os.path.join(carpeta, f"dataset_{year}.csv.gz")
    df.to_csv(output_path, index=False, compression='gzip')
    print(f"  💾 Guardado: {output_path}")

    return df.shape


# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    resumen = {}
    for year in YEARS:
        shape = procesar_año(year)
        resumen[year] = shape if shape else "❌ FALLIDO"

    print(f"\n{'='*60}")
    print("📊 RESUMEN FINAL")
    print("="*60)
    for year, shape in resumen.items():
        print(f"  {year}: {shape}")
    print("\n¡Ahora utilizar el fichero testCalidadCSV.ipynb para corroborar los datos!")



🔬 Procesando año 2020
  📂 Cargando instant...
     Variables: ['u100', 'v100', 'tcc', 'u10', 'v10', 't2m', 'sp']
     Timestamps: 8784
     ✅ solar tcc ponderado — (8784, 2)
     ✅ solar t2m ponderado — (8784, 2)
     ✅ eólica u100 ponderado — (8784, 2)
     ✅ eólica v100 ponderado — (8784, 2)
     ✅ eólica u10 ponderado — (8784, 2)
     ✅ eólica v10 ponderado — (8784, 2)
     ✅ eólica sp ponderado — (8784, 2)
     ✅ eólica t2m ponderado — (8784, 2)
     ✅ 47 capitales extraídas — (8784, 48)
     Suma pesos poblacionales 2020: 1.0000 (✅)
     ✅ HDD/CDD peninsular (2020) — (8784, 4)
  📂 Cargando accum...
     Timestamps accum año 2020: 8784
     Variables: ['tp', 'ssrd']
     ✅ solar ssrd ponderado — (8784, 2)
     ✅ hidráulica tp ponderado — (8784, 2)

  📊 Tras merge — shape: (8784, 14)
  🕐 Timestamps únicos: 8784
  📅 Rango: 2020-01-01 00:00:00 → 2020-12-31 23:00:00
  ✅ densidad_aire_eolica calculada
     Rango: [1.091, 1.226] kg/m³
  ✅ wind_power_density_ponderado calculado
  ✅ Shape

##### instant → 8760 filas (8784 en bisiestos)
##### accum   → 8760 filas (tras stack + filtro año)
##### merge   → 8760 filas × 11 columnas finales

#### Columnas:
  ###### time, tcc_solar, t2m_celsius_solar, ssrd_solar,
  ###### tp_hidraulica, wind_speed_100m_eolica,
  ###### wind_speed_10m_eolica, wind_shear_eolica,
  ###### t2m_celsius_demanda, HDD, CDD

### Para ver el output en su totalidad:
1. Abre la configuración con el atajo Ctrl + , (Windows/Linux) o Cmd + , (Mac).
2. En la barra de búsqueda superior, escribe: notebook output.
3. Busca la opción llamada: Notebook > Output: Text Line Limit.
4. Verás que por defecto tiene un límite (normalmente 30 o 100 líneas). Cambia ese número a uno mucho mayor (por ejemplo, 5000).